In [ ]:
import torch
import torch.nn as nn
import time
import matplotlib.pyplot as plt

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader


# --------------------------------------------------
# PARAMETERS
# --------------------------------------------------

BATCH_SIZE = 8
EPOCHS = 5
NUM_CLASSES = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.backends.cudnn.benchmark = True


# --------------------------------------------------
# TRANSFORMS
# --------------------------------------------------

train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Grayscale(3),  # convert 1-channel → 3-channel
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Grayscale(3),
    transforms.ToTensor(),
])


# --------------------------------------------------
# DATASETS
# --------------------------------------------------

train_ds = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=train_tf
)

val_ds = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=val_tf
)

train_loader = DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True)
val_loader   = DataLoader(val_ds,batch_size=BATCH_SIZE)

print("Train samples:", len(train_ds))
print("Validation samples:", len(val_ds))


# --------------------------------------------------
# MODEL
# --------------------------------------------------

def load_model():

    model = models.googlenet(
        weights=models.GoogLeNet_Weights.DEFAULT
    )

    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

    return model.to(device)

model = load_model()

print("Device:", device)


# --------------------------------------------------
# TRAINING SETUP
# --------------------------------------------------

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)


# --------------------------------------------------
# EVALUATION
# --------------------------------------------------

def evaluate():

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for imgs,labels in val_loader:

            imgs = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs)

            preds = outputs.argmax(1)

            correct += (preds==labels).sum().item()
            total += labels.size(0)

    return correct/total


# --------------------------------------------------
# TRAINING LOOP
# --------------------------------------------------

def train():
    train_losses = []
    val_accs = []

    for epoch in range(EPOCHS):

        model.train()

        running_loss = 0
        t0 = time.time()

        for imgs,labels in train_loader:

            imgs = imgs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)

            outputs = model(imgs)

            loss = criterion(outputs,labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        train_losses.append(running_loss)

        val_acc = evaluate()

        val_accs.append(val_acc)

        elapsed = time.time() - t0

        print(
            f"Epoch {epoch+1}/{EPOCHS} | "
            f"loss={running_loss:.4f} | "
            f"val_acc={val_acc:.4f} | "
            f"time={elapsed:.1f}s"
        )

    print("Training finished.")
    
    return train_losses, val_accs


if __name__ == "__main__":
    train_losses, val_accs = train()
    
    # Plot results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.plot(train_losses, label="train_loss")
    ax1.set_title("Training Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True)
    
    ax2.plot(val_accs, label="val_accuracy")
    ax2.set_title("Validation Accuracy")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()

ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host